# QazRuFake v4: XLM-R with class weights and more seeds; stronger zero-shot LLM

1. *Runtime → Change runtime type → **T4 GPU*** (or better).
2. Drag **`data/qazrufake.jsonl`** into the Files panel on the left (📁) and wait for the upload to finish.
3. *Runtime → **Run all***. Allow Google Drive access when asked.

**Part A** retrains XLM-R base with a **class-weighted loss**: 5 seeds for every protocol row, 10 seeds for the two least stable rows (Russian same-publisher claim-only and QazRuFake-Core). About 2 hours on a T4.
**Part B** runs a larger zero-shot LLM (default **Qwen2.5-14B-Instruct**, 4-bit) with the prompt **in the language of the text** (Russian or Kazakh), on **dev and test** items, so that the decision threshold can be chosen on dev. The QazRuFake-Core and Russian same-publisher items go first. About 2–3 hours; partial results are already usable.

Both parts skip work that is already saved, so after a disconnect just press *Run all* again.
When it finishes (or when you stop it), send back from Google Drive: **`qazrufake_preds_v4.jsonl`**, **`qazrufake_preds_llm_v4.jsonl`** and **`qazrufake_v4_summary.json`**.


In [ ]:
# 1. GPU check, packages, Drive, corpus
import torch, os, json, shutil
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch Runtime -> T4 GPU, then rerun')
!pip install -q transformers datasets accelerate scikit-learn bitsandbytes
from google.colab import drive, files
drive.mount('/content/drive')
D = '/content/drive/MyDrive/'
CORPUS = next((p for p in ['qazrufake.jsonl', 'qazrufake_full.jsonl', D + 'qazrufake.jsonl'] if os.path.exists(p)), None)
if CORPUS is None:
    print('Corpus not found: choose data/qazrufake.jsonl in the upload dialog below')
    up = files.upload()
    CORPUS = next(iter(up))
if not CORPUS.startswith(D):
    shutil.copy(CORPUS, D + 'qazrufake.jsonl')  # keep a copy on Drive in case the runtime restarts
R = [json.loads(l) for l in open(CORPUS, encoding='utf-8')]
assert all(r.get('text') for r in R), 'Some items have no text'
OUT = D + 'qazrufake_preds_v4.jsonl'
OUT_LLM = D + 'qazrufake_preds_llm_v4.jsonl'
print(len(R), 'items')

## Part A. XLM-R with a class-weighted loss

In [ ]:
# 2. Settings (the same training sets as in the paper)
ru = lambda r: r['lang'] == 'ru'
kk = lambda r: r['lang'] == 'kk'
rufc = lambda r: r['lang'] == 'ru' and r['source_name'] != 'Tengrinews.kz'
kkfc = lambda r: r['lang'] == 'kk' and r['fc_kk']
kkcore = lambda r: r['lang'] == 'kk' and r['unamb']
SETTINGS = {'ru|text': (ru, 'text'), 'kk|text': (kk, 'text'),
            'rufc|text': (rufc, 'text'), 'rufc|claim': (rufc, 'claim'),
            'kkfc|text': (kkfc, 'text'), 'kkfc|claim': (kkfc, 'claim'),
            'kkunamb|claim': (kkcore, 'claim')}
SEEDS = {s: [1, 2, 3, 4, 5] for s in SETTINGS}
SEEDS['rufc|claim'] = SEEDS['kkunamb|claim'] = list(range(1, 11))
MODEL = 'xlm-roberta-base'
print(sum(len(v) for v in SEEDS.values()), 'runs')

In [ ]:
# 3. Training function: class-weighted cross-entropy (weights = n / (2 * n_class) on the training set)
import shutil, numpy as np
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **k):
        super().__init__(*a, **k); self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits.float(), labels, weight=self.cw.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def done(setting, seed):
    if not os.path.exists(OUT): return False
    key = f'"setting": "{setting}", "seed": {seed},'
    return any(key in l for l in open(OUT))

def run(setting, seed, epochs=4, lr=2e-5, bs=16, maxlen=256):
    if done(setting, seed):
        print('skip (already done):', setting, seed); return
    filt, field = SETTINGS[setting]
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL)
    def ds(rs):
        d = Dataset.from_dict({'text': [r[field] for r in rs], 'label': [r['label'] for r in rs]})
        return d.map(lambda b: tok(b['text'], truncation=True, max_length=maxlen), batched=True)
    tr = [r for r in R if r['split'] == 'train' and filt(r)]
    dv = [r for r in R if r['split'] == 'dev' and filt(r)]
    te = [r for r in R if r['split'] == 'test']
    n1 = sum(r['label'] for r in tr); n0 = len(tr) - n1
    cw = torch.tensor([len(tr) / (2 * n0), len(tr) / (2 * n1)], dtype=torch.float)
    dtr, ddv = ds(tr), ds(dv)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    def metrics(p):
        y = p.label_ids; pr = p.predictions.argmax(-1)
        return {'f1m': f1_score(y, pr, average='macro'), 'acc': accuracy_score(y, pr)}
    args = TrainingArguments(output_dir='out', learning_rate=lr, per_device_train_batch_size=bs,
        per_device_eval_batch_size=64, num_train_epochs=epochs, weight_decay=0.01,
        warmup_steps=int(0.1 * epochs * -(-len(dtr) // bs)),
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='f1m', save_total_limit=1, fp16=torch.cuda.is_available(),
        report_to='none', seed=seed)
    t = WeightedTrainer(model=model, args=args, train_dataset=dtr, eval_dataset=ddv,
                        data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics, class_weights=cw)
    t.train()
    p = t.predict(ds(te))
    prob = torch.softmax(torch.tensor(p.predictions).float(), -1)[:, 1].numpy()
    with open(OUT, 'a') as f:
        for r, q in zip(te, prob):
            f.write(json.dumps({'model': MODEL, 'loss': 'class_weighted', 'setting': setting, 'seed': seed, 'uid': r['uid'],
                                'test': r['lang'], 'prob': round(float(q), 5)}) + '\n')
    y = np.array([r['label'] for r in te]); pr = (prob > 0.5).astype(int)
    m = np.array([filt(r) for r in te])
    print(setting, seed, 'train REAL/FAKE = %d/%d' % (n0, n1), 'F1m on own test subset = %.3f' % f1_score(y[m], pr[m], average='macro'),
          'share predicted FAKE = %.2f' % pr[m].mean())
    del t, model; torch.cuda.empty_cache(); shutil.rmtree('out', ignore_errors=True)

In [ ]:
# 4. Run Part A (the two least stable rows first)
ORDER = ['rufc|claim', 'kkunamb|claim', 'rufc|text', 'kkfc|claim', 'kkfc|text', 'ru|text', 'kk|text']
for s in ORDER:
    for seed in SEEDS[s]:
        print('=' * 20, s, seed)
        run(s, seed)
print('PART A DONE')

## Part B. Zero-shot LLM: larger model, prompt in the language of the text, threshold chosen on dev

In [ ]:
# 5. Load the LLM in 4-bit. If it does not fit in memory, set LLM = 'Qwen/Qwen2.5-7B-Instruct'
import gc; gc.collect(); torch.cuda.empty_cache()
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
LLM = 'Qwen/Qwen2.5-14B-Instruct'
ltok = AutoTokenizer.from_pretrained(LLM, padding_side='left')
llm = AutoModelForCausalLM.from_pretrained(LLM, device_map='auto', low_cpu_mem_usage=True,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                               bnb_4bit_quant_type='nf4'))
llm.eval()
SYSTEM = {
 'ru': 'Вы — внимательный фактчекер, проверяющий новости и сообщения в социальных сетях Казахстана.',
 'kk': 'Сіз Қазақстандағы жаңалықтар мен әлеуметтік желілердегі хабарламаларды тексеретін мұқият фактчекерсіз.'}
USER = {
 'ru': ('Ниже приведён текст, который сообщает об утверждении или новости из Казахстана.\n'
        'Определите, является ли сообщаемое правдой (REAL) или ложью либо манипуляцией (FAKE).\n'
        'Ответьте ровно одним словом: REAL или FAKE.\n\nТекст:\n'),
 'kk': ('Төмендегі мәтін Қазақстандағы бір мәлімдеме немесе жаңалық туралы хабарлайды.\n'
        'Онда айтылған ақпарат шындық па (REAL), әлде жалған не адастыратын ба (FAKE), соны анықтаңыз.\n'
        'Тек бір сөзбен жауап беріңіз: REAL немесе FAKE.\n\nМәтін:\n')}
def prompt(text, lang):
    msgs = [{'role': 'system', 'content': SYSTEM[lang]}, {'role': 'user', 'content': USER[lang] + text}]
    return ltok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
ID_REAL = ltok.encode('REAL', add_special_tokens=False)[0]
ID_FAKE = ltok.encode('FAKE', add_special_tokens=False)[0]
assert ID_REAL != ID_FAKE
print('answer tokens', ltok.decode([ID_REAL]), ltok.decode([ID_FAKE]))

In [ ]:
# 6. Score dev and test items. Order: Core claim-only, Russian same-publisher claim-only, then the rest.
done_llm = set()
if os.path.exists(OUT_LLM):
    done_llm = {(d['field'], d['uid']) for d in map(json.loads, open(OUT_LLM)) if d['model'] == LLM}
items = [r for r in R if r['split'] in ('dev', 'test')]
prio = [('claim', lambda r: r['lang'] == 'kk' and r['unamb']), ('claim', rufc), ('text', rufc),
        ('claim', kkfc), ('text', kkfc), ('text', lambda r: True), ('claim', lambda r: True)]
jobs, seen = [], set(done_llm)
for f, filt in prio:
    for r in items:
        if filt(r) and (f, r['uid']) not in seen:
            jobs.append((f, r)); seen.add((f, r['uid']))
print(len(jobs), 'prompts to run')
BS = 4
with torch.no_grad():
    for i in range(0, len(jobs), BS):
        batch = jobs[i:i + BS]
        enc = ltok([prompt(r[f][:2000], r['lang']) for f, r in batch], return_tensors='pt', padding=True).to(llm.device)
        logits = llm(**enc).logits[:, -1, :].float()
        two = torch.softmax(logits[:, [ID_REAL, ID_FAKE]], -1)[:, 1].cpu().numpy()
        with open(OUT_LLM, 'a') as fo:
            for (f, r), p in zip(batch, two):
                fo.write(json.dumps({'model': LLM, 'prompt': 'native', 'field': f, 'uid': r['uid'], 'split': r['split'],
                                     'prob': round(float(p), 5)}) + '\n')
        if (i // BS) % 50 == 0: print(i + len(batch), '/', len(jobs))
print('PART B DONE')

In [ ]:
# 7. Summary (runs with whatever is saved)
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, roc_auc_score
by = {r['uid']: r for r in R}
SUB = {'ru|all': ru, 'ru|fcout': rufc, 'kk|all': kk, 'kk|fc': kkfc, 'kk|unamb': kkcore}
summary = {}
if os.path.exists(OUT):
    X = pd.DataFrame(map(json.loads, open(OUT)))
    OWN = {'ru|text': 'ru|all', 'kk|text': 'kk|all', 'rufc|text': 'ru|fcout', 'rufc|claim': 'ru|fcout',
           'kkfc|text': 'kk|fc', 'kkfc|claim': 'kk|fc', 'kkunamb|claim': 'kk|unamb'}
    for s, sub in OWN.items():
        vals, fakes = [], []
        for seed, d in X[X.setting == s].groupby('seed'):
            d = d[d.uid.map(lambda u: SUB[sub](by[u]))]
            y = d.uid.map(lambda u: by[u]['label']).values; pr = (d.prob.values > 0.5).astype(int)
            vals.append(f1_score(y, pr, average='macro')); fakes.append(pr.mean())
        if vals:
            summary[f'xlmr_cw|{s}|{sub}'] = {'n_seeds': len(vals), 'f1_mean': float(np.mean(vals)),
                'f1_sd': float(np.std(vals, ddof=1)) if len(vals) > 1 else None,
                'f1_per_seed': [round(v, 3) for v in vals], 'collapsed_seeds': int(sum(f in (0.0, 1.0) for f in fakes))}
if os.path.exists(OUT_LLM):
    L = pd.DataFrame(map(json.loads, open(OUT_LLM)))
    for f in ['text', 'claim']:
        for sname, filt in SUB.items():
            d = L[(L.field == f) & L.uid.map(lambda u: filt(by[u]))]
            dv, te = d[d.split == 'dev'], d[d.split == 'test']
            if len(te) == 0 or len(dv) == 0: continue
            ydv = dv.uid.map(lambda u: by[u]['label']).values; yte = te.uid.map(lambda u: by[u]['label']).values
            grid = np.unique(np.quantile(dv.prob.values, np.linspace(0.01, 0.99, 99)))
            thr = max(grid, key=lambda t: f1_score(ydv, (dv.prob.values > t).astype(int), average='macro'))
            summary[f'llm|{f}|{sname}'] = {'n_test': len(te), 'auc': float(roc_auc_score(yte, te.prob)),
                'f1m_thr0.5': float(f1_score(yte, (te.prob.values > 0.5).astype(int), average='macro')),
                'thr_dev': float(thr), 'f1m_thr_dev': float(f1_score(yte, (te.prob.values > thr).astype(int), average='macro'))}
json.dump(summary, open(D + 'qazrufake_v4_summary.json', 'w'), indent=1)
print(pd.DataFrame(summary).T.to_string())